# Séance 2 — Régression linéaire : du modèle statistique à l'évaluation

## Fil rouge : Diabetes

Nous allons maintenant construire notre premier modèle.

Objectifs :
- comprendre le modèle linéaire ;
- comprendre ce que sont les paramètres ;
- comprendre la fonction de perte ;
- ajuster une régression avec `scikit-learn` ;
- calculer MSE, RMSE, MAE et R² ;
- analyser les coefficients avec `statsmodels` ;
- étudier leur significativité.

## 1. Le modèle linéaire

Avec une variable :

$
Y = \beta_0 + \beta_1X + \varepsilon
$

Avec plusieurs variables :

$
Y =
\beta_0+
\beta_1X_1+
\cdots+
\beta_pX_p+
\varepsilon
$

Les $\beta_j$ sont les **paramètres** du modèle.

Le modèle produit une prédiction :

$
\hat Y =
\beta_0+
\beta_1X_1+
\cdots+
\beta_pX_p
$

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

diabetes = load_diabetes()

X = diabetes.data
y = diabetes.target

Xtr, Xte, ytr, yte = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(Xtr.shape)
print(Xte.shape)

## 2. Pourquoi séparer train et test ?

Le modèle doit apprendre sur certaines observations puis être évalué sur des observations qu'il n'a pas utilisées pour apprendre.

- `Xtr`, `ytr` : apprentissage ;
- `Xte`, `yte` : évaluation finale.

C'est une première façon d'étudier la **généralisation**.

## 3. Ajuster le modèle

Avec `scikit-learn` :

```python
modele.fit(Xtr, ytr)
```

fait apprendre les paramètres du modèle à partir des données.

In [ ]:
modele = LinearRegression()

modele.fit(Xtr, ytr)

In [ ]:
print("Intercept :", modele.intercept_)

for nom, coef in zip(diabetes.feature_names, modele.coef_):
    print(f"{nom:>5} : {coef:8.3f}")

### Que fait réellement `fit()` ?

Le modèle cherche les coefficients qui rendent les prédictions aussi proches que possible des observations d'entraînement.

Pour la régression linéaire, on utilise classiquement la somme des carrés des erreurs :

$
\sum_{i=1}^n (y_i-\hat y_i)^2
$

C'est l'origine de la **moindre carrés**.

## 4. La fonction de perte

Une façon de mesurer l'erreur moyenne est :

$
MSE =
\frac{1}{n}
\sum_{i=1}^n
(y_i-\hat y_i)^2
$

Pourquoi mettre au carré ?

- les erreurs positives et négatives ne s'annulent pas ;
- les grosses erreurs sont davantage pénalisées.

In [ ]:
ytr_pred = modele.predict(Xtr)
yte_pred = modele.predict(Xte)

mse_train = mean_squared_error(ytr, ytr_pred)
mse_test = mean_squared_error(yte, yte_pred)

print("MSE train :", mse_train)
print("MSE test  :", mse_test)

## 5. Les critères d'évaluation

### MSE

$
MSE = \frac1n\sum(y_i-\hat y_i)^2
$

Plus petit = meilleur.

### RMSE

$
RMSE = \sqrt{MSE}
$

Il est exprimé dans l'unité de la variable cible.

### MAE

$
MAE = \frac1n\sum|y_i-\hat y_i|
$

### R²

$
R^2 =
1 -
\frac{\sum(y_i-\hat y_i)^2}
{\sum(y_i-\bar y)^2}
$

Le dénominateur correspond à l'erreur obtenue si l'on prédit toujours la moyenne de `y`.

In [ ]:
rmse_train = np.sqrt(mean_squared_error(ytr, ytr_pred))
rmse_test = np.sqrt(mean_squared_error(yte, yte_pred))

mae_train = mean_absolute_error(ytr, ytr_pred)
mae_test = mean_absolute_error(yte, yte_pred)

r2_train = r2_score(ytr, ytr_pred)
r2_test = r2_score(yte, yte_pred)

print(f"Train : MSE={mse_train:.2f}, RMSE={rmse_train:.2f}, MAE={mae_train:.2f}, R²={r2_train:.3f}")
print(f"Test  : MSE={mse_test:.2f}, RMSE={rmse_test:.2f}, MAE={mae_test:.2f}, R²={r2_test:.3f}")

### Question d'interprétation

Comparez les performances train et test.

- Les scores sont-ils proches ?
- Que peut-on dire sur le surapprentissage dans ce cas ?
- Pourquoi le R² du test peut-il être plus faible que celui du train ?

## 6. Visualiser prédictions et observations

In [ ]:
plt.figure(figsize=(6, 6))

plt.scatter(yte, yte_pred)

minimum = min(yte.min(), yte_pred.min())
maximum = max(yte.max(), yte_pred.max())

plt.plot(
    [minimum, maximum],
    [minimum, maximum],
    linestyle="--"
)

plt.xlabel("Valeurs observées")
plt.ylabel("Valeurs prédites")
plt.title("Prédictions vs observations")
plt.show()

Si le modèle prédisait parfaitement, tous les points seraient sur la droite :

$
\hat y = y
$

Plus les points s'en éloignent, plus les erreurs sont importantes.

## 7. Une approche statistique avec `statsmodels`

`scikit-learn` est particulièrement pratique pour construire des modèles prédictifs.

`statsmodels` fournit davantage d'outils pour l'inférence statistique :
- erreurs standards ;
- statistiques de test ;
- p-values ;
- intervalles de confiance.

In [ ]:
import statsmodels.api as sm

Xtr_df = pd.DataFrame(
    Xtr,
    columns=diabetes.feature_names
)

Xtr_sm = sm.add_constant(Xtr_df)

modele_sm = sm.OLS(ytr, Xtr_sm).fit()

print(modele_sm.summary())

## 8. Significativité des coefficients

Pour chaque coefficient, on peut tester :

$
H_0 : \beta_j=0
$

contre une hypothèse alternative.

La colonne `P>|t|` donne la p-value du test.

Une p-value faible apporte un argument contre l'hypothèse $\beta_j=0$.

⚠️ Cela ne signifie pas :

> « cette variable est importante pour prédire ».

La significativité statistique et la performance prédictive sont deux notions différentes.

In [ ]:
resume = pd.DataFrame({
    "nom": modele_sm.params.index,
    "coef": modele_sm.params.to_numpy(),
    "std_err": modele_sm.bse.to_numpy(),
    "t": modele_sm.tvalues.to_numpy(),
    "p_value": modele_sm.pvalues.to_numpy()
})

resume

## 9. Intervalles de confiance

Pour chaque coefficient, `statsmodels` fournit un intervalle de confiance.

In [ ]:
conf = modele_sm.conf_int()

resume_ic = pd.DataFrame({
    "coef": modele_sm.params.to_numpy(),
    "IC_2.5%": conf.iloc[:, 0].to_numpy(),
    "IC_97.5%": conf.iloc[:, 1].to_numpy()
}, index=modele_sm.params.index)

resume_ic

### Question d'interprétation

Pour les coefficients des variables explicatives :

1. Quels coefficients semblent statistiquement différents de 0 ?
2. Les intervalles de confiance contiennent-ils 0 pour ces variables ?
3. Comparez vos conclusions avec les corrélations de la séance 1.

Attention : dans la régression multiple, un coefficient mesure l'effet associé à une variable **à variables explicatives restantes fixées**.

## 10. Une seule variable : `bmi`

La variable `bmi` était fortement associée à `target`.

Construisons maintenant une régression simple.

In [ ]:
bmi = df_bmi = pd.DataFrame(
    diabetes.data[:, [diabetes.feature_names.index("bmi")]],
    columns=["bmi"]
)

Xbmi_tr, Xbmi_te, ybmi_tr, ybmi_te = train_test_split(
    bmi,
    y,
    test_size=0.2,
    random_state=42
)

modele_bmi = LinearRegression()
modele_bmi.fit(Xbmi_tr, ybmi_tr)

pred_bmi = modele_bmi.predict(Xbmi_te)

print("Coefficient :", modele_bmi.coef_[0])
print("R² test :", modele_bmi.score(Xbmi_te, ybmi_te))

### Question finale

La variable `bmi` seule permet-elle de prédire correctement `target` ?

Pourquoi le modèle à 10 variables peut-il être meilleur qu'un modèle avec `bmi` seul ?

# Bilan

Nous avons maintenant les éléments fondamentaux de la chaîne :

**Données**

↓

**Modèle linéaire**

↓

**Prédictions**

↓

**Fonction de perte : MSE**

↓

**Optimisation des paramètres**

↓

**Évaluation : MSE / RMSE / MAE / R²**

Nous avons également vu qu'une approche statistique permet d'aller plus loin avec :
- erreurs standards ;
- p-values ;
- intervalles de confiance.

La prochaine séance sera consacrée à la **comparaison de plusieurs modèles** et à la **validation croisée**.